# Практика · Голови, позиції, маски

> ⏱ **Зошит навчає девʼятнадцять маленьких трансформерів.** Заміряно: близько
> **240 с процесорного часу** (чотири хвилини) на чотирьох ядрах без відеокарти,
> в один потік. За стінним годинником на вільній машині — **259 с**; на
> зайнятій буде більше, і це нормально. Якщо здається, що все зависло, — ні,
> іде навчання.

Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
Домашнє: [homework.html](homework.html)

## Задача, яку ми тут розвʼязуємо

Ми будуємо **мовну модель на трансформері** для української мови й перевіряємо
замірами три твердження про її будову. Дані справжні: українські переклади
інтерфейсів, які вже лежать у системі.

Твердження, які перевіримо:

1. **Чи працюють кілька голів уваги краще за одну** при однаковій кількості
   параметрів. Заміряємо перплексію на **пʼятьох зернах** і дивимось, чи
   перетинаються купи. Пʼять, а не три: на трьох зернах розкид виходить оманливо
   вузьким, і різниця здається більшою, ніж вона є. Відповідь наперед не відома —
   вона залежить від того, скільки даних ми собі дозволимо.
2. **Self-attention не бачить порядку слів.** Це доводиться без навчання: досить
   переставити слова на вході й порівняти вихід число в число.
3. **Без причинної маски модель бачить відповідь** — і перплексія обвалюється,
   бо задача перетворюється на копіювання.

Дорогою напишемо багатоголову увагу руками й звіримо її з бібліотечною.

**Зошит самодостатній:** лекцію читати не обовʼязково, усе потрібне пояснено тут.

In [ ]:
import os
# ці чотири рядки мусять стояти ДО імпорту numpy і torch: без них потоки OpenMP
# крутяться в очікуванні, і process_time() рахує це очікування як роботу
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import sys, re, glob, gettext, math, random, time, itertools, gc
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)          # те саме для torch, уже після імпорту
NOTEBOOK_START = time.process_time()

print('python  ', sys.version.split()[0])
print('numpy   ', np.__version__)
print('torch   ', torch.__version__)
print('потоків ', torch.get_num_threads())

## Крок 1 · Звідки беремо текст

Корпус — українські переклади інтерфейсів, які лежать у кожній системі Linux
у файлах `/usr/share/locale/uk/LC_MESSAGES/*.mo`. Це справжня українська мова,
написана людьми, а не згенерована формулою.

Що варто знати про цей текст одразу: він **вузький за доменом**. Технічна
лексика, короткі речення, багато наказового способу й повідомлень про помилки.
Висновки про українську мову взагалі з нього робити не можна — тільки про цей
різновид тексту.

Запасного синтетичного корпусу тут навмисно немає. Запобіжник, який ніколи не
пробували, гірший за його відсутність: із ним падіння станеться на двадцятій
клітинці після кількох хвилин рахунку, а без нього помилка видно одразу й
зрозуміла.

In [ ]:
def load_documents():
    """Читає всі українські каталоги перекладів, що є на цій машині."""
    documents = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
            for source, translation in catalog._catalog.items():
                # беремо лише довші рядки: короткі — це підписи кнопок, не речення
                if isinstance(source, str) and isinstance(translation, str) \
                   and len(translation) > 30 and 'Project-Id' not in translation:
                    documents.append(translation)
        except Exception:
            pass                     # зіпсований чи чужий формат — просто пропускаємо
    return documents

documents = load_documents()
print('знайдено документів:', len(documents))

if len(documents) < 5000:
    raise RuntimeError(
        'Українських перекладів на цій машині замало (%d документів).\n'
        'Постав українську локаль: у Fedora `sudo dnf install langpacks-uk`,\n'
        'в Ubuntu `sudo apt install language-pack-uk`. Без корпусу зошит\n'
        'рахувати нічого.' % len(documents))

print('приклад:', documents[17][:90])

## Крок 2 · Ріжемо текст на слова

Токенізатор — той самий регулярний вираз, яким користуються всі теми цього курсу.
Він трактує апостроф як **звʼязку всередині слова**: `зʼєднання` — один токен, а не
два. Якщо взяти інший вираз, усі числа курсу перестануть сходитися між темами, тож
беремо канонічний.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"    # канон курсу, після lowercase
_token_re = re.compile(TOKEN_PATTERN)

def tokenize(text):
    return _token_re.findall(text.lower())

sentences = [tokenize(d) for d in documents]
lengths = np.array([len(s) for s in sentences])

print('слововживань   ', int(lengths.sum()))
print('словоформ      ', len({w for s in sentences for w in s}))
print('медіана довжини', int(np.median(lengths)), 'слів')
print('90-й перцентиль', int(np.percentile(lengths, 90)), 'слів')
print('приклад токенів:', sentences[17][:8])

## Крок 3 · Три вибірки, а не дві

Звична схема — навчальна й перевірна вибірки. Нам потрібна **третя**, і ось чому.

Далі ми підбиратимемо швидкість навчання. Якщо перебирати її й щоразу дивитись на
перевірну вибірку, ми підглядаємо у відповідь: обраний варіант виявиться найкращим
саме на цих даних. Тому з навчальної частини заздалегідь відрізаємо **відкладену**
вибірку: на ній перебираємо, а перевірну чіпаємо один-єдиний раз наприкінці.

Слова, які трапились у навчальній частині менше пʼяти разів, у словник не беремо —
на них однаково нічого не вивчиш. Замість них ставимо спецтокен `<unk>`.

In [ ]:
kept = [s for s in sentences if 2 <= len(s) <= 30]

shuffled = list(kept)
random.Random(0).shuffle(shuffled)            # зерно 0: поділ однаковий у всіх
n_train = int(len(shuffled) * 0.9)
train_words, val_words = shuffled[:n_train], shuffled[n_train:]

index = list(range(len(train_words)))
random.Random(11).shuffle(index)
n_holdout = int(len(train_words) * 0.05)
holdout_words = [train_words[i] for i in index[:n_holdout]]
fit_words = [train_words[i] for i in index[n_holdout:]]

MIN_COUNT = 5
counts = Counter(w for s in fit_words for w in s)
vocab = ['<pad>', '<eos>', '<unk>'] + sorted(w for w, c in counts.items() if c >= MIN_COUNT)
word_to_id = {w: i for i, w in enumerate(vocab)}
PAD, EOS, UNK = 0, 1, 2
V = len(vocab)

def encode(sents):
    return [[word_to_id.get(w, UNK) for w in s] for s in sents]

fit, holdout, val = encode(fit_words), encode(holdout_words), encode(val_words)

print('навчальних речень  ', len(fit))
print('відкладених        ', len(holdout))
print('перевірних         ', len(val))
print('словник            ', V, '(разом із <pad>, <eos>, <unk>)')
print('частка <unk> у перевірній %.4f'
      % (sum(1 for s in val for w in s if w == UNK) / sum(len(s) for s in val)))

## Крок 4 · Модель

Мовна модель на трансформері складається з чотирьох частин:

1. **ембединги** — таблиця, яка перетворює номер слова на вектор із 128 чисел;
2. **позиційне кодування** — вектор, який залежить від номера позиції й додається
   до ембедингу; рахується формулою з синусів і косинусів, нічого не вчить;
3. **два шари трансформера** — у кожному багатоголова увага й повнозвʼязна
   надбудова;
4. **вихідний шар** — перетворює вектор назад на оцінки для кожного слова словника.

Три перемикачі в конструкторі — це наші три досліди: `nhead` (скільки голів),
`use_pos` (чи додавати позиційне кодування), а причинну маску вмикає аргумент
`causal` у `forward`.

In [ ]:
D_MODEL, N_LAYERS, FF = 128, 2, 512

def sinusoidal_table(max_len, d_model):
    """Таблиця позиційних кодувань: пара координат = одна стрілка годинника."""
    position = torch.arange(max_len).unsqueeze(1).float()
    even = torch.arange(0, d_model, 2).float()
    speed = torch.exp(-math.log(10000.0) * even / d_model)   # швидкість стрілки
    table = torch.zeros(max_len, d_model)
    table[:, 0::2] = torch.sin(position * speed)
    table[:, 1::2] = torch.cos(position * speed)
    return table

class TransformerLM(nn.Module):
    def __init__(self, vocab_size, nhead=4, use_pos=True, max_len=64):
        super().__init__()
        self.use_pos = use_pos
        self.nhead = nhead
        self.embedding = nn.Embedding(vocab_size, D_MODEL, padding_idx=PAD)
        self.register_buffer('pos', sinusoidal_table(max_len, D_MODEL))
        layer = nn.TransformerEncoderLayer(D_MODEL, nhead, FF, dropout=0.0,
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, N_LAYERS,
                                             norm=nn.LayerNorm(D_MODEL))
        self.output = nn.Linear(D_MODEL, vocab_size)

    def forward(self, x, causal=True, pad_mask=True):
        steps = x.shape[1]
        # множник √d вирівнює масштаб ембедингів із масштабом кодування позицій
        h = self.embedding(x) * math.sqrt(D_MODEL)
        if self.use_pos:
            h = h + self.pos[:steps].unsqueeze(0)
        forbid = torch.triu(torch.ones(steps, steps, dtype=torch.bool), diagonal=1) \
                 if causal else None
        h = self.encoder(h, mask=forbid,
                         src_key_padding_mask=(x == PAD) if pad_mask else None)
        return self.output(h)

for nhead in (1, 2, 4, 8):
    total = sum(p.numel() for p in TransformerLM(V, nhead=nhead).parameters())
    print('голів %d -> координат на голову %3d -> параметрів %d'
          % (nhead, D_MODEL // nhead, total))

Зверни увагу на останній стовпчик: **кількість параметрів однакова**. Голови не
додають ваг, вони ділять ті самі. Далі ми з цього й виходимо: порівнюємо не «дорожчу
модель із дешевшою», а два розкрої одного бюджету.

## Крок 5 · Пишемо багатоголову увагу руками

Перш ніж чогось навчати, переконаймося, що всередині бібліотечного шару немає магії.
Напишемо ту саму операцію самі, з нуля, і звіримо результати.

Порядок дій усередині: із кожного вектора роблять три — запит, ключ і значення;
ріжуть їх на голови; у кожній голові рахують збіги запитів із ключами; забороненим
клітинкам ставлять мінус нескінченність; softmax перетворює збіги на ваги; ваги
змішують значення; голови склеюють і множать на вихідну матрицю.

In [ ]:
def manual_multihead(mha, x, causal=True, pad_mask=None):
    """Багатоголова увага руками. Ваги беремо з готового шару, щоб було що звіряти."""
    d = mha.embed_dim
    heads = mha.num_heads
    d_head = d // heads
    batch, steps, _ = x.shape

    # у torch три проєкції лежать однією матрицею одна під одною
    W_query, W_key, W_value = mha.in_proj_weight.split(d, dim=0)
    b_query, b_key, b_value = mha.in_proj_bias.split(d, dim=0)

    # ріжемо на голови: (пачка, крок, голова, координата) -> (пачка, голова, крок, ...)
    def to_heads(t):
        return t.view(batch, steps, heads, d_head).transpose(1, 2)

    query = to_heads(x @ W_query.T + b_query)
    key = to_heads(x @ W_key.T + b_key)
    value = to_heads(x @ W_value.T + b_value)

    # ділимо на корінь із розміру ГОЛОВИ, а не всього вектора
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_head)

    if causal:
        # заборона дивитись праворуч від себе
        future = torch.triu(torch.ones(steps, steps, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(future, float('-inf'))
    if pad_mask is not None:
        # заборона читати порожні місця; форма (пачка, 1, 1, крок) розтягнеться сама
        scores = scores.masked_fill(pad_mask[:, None, None, :], float('-inf'))

    weights = torch.softmax(scores, dim=-1)
    mixed = (weights @ value).transpose(1, 2).reshape(batch, steps, d)
    return mixed @ mha.out_proj.weight.T + mha.out_proj.bias, weights

torch.manual_seed(0)
probe = nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.0)
x_probe = torch.randn(3, 7, D_MODEL)
pad_probe = torch.zeros(3, 7, dtype=torch.bool)
pad_probe[0, 6:] = True          # у першому реченні останнє місце - доповнення
pad_probe[1, 5:] = True          # у другому - два останні
future_probe = torch.triu(torch.ones(7, 7, dtype=torch.bool), diagonal=1)

ours, weights_ours = manual_multihead(probe, x_probe, True, pad_probe)
theirs, weights_theirs = probe(x_probe, x_probe, x_probe, attn_mask=future_probe,
                               key_padding_mask=pad_probe, need_weights=True,
                               average_attn_weights=False)

print('найбільша розбіжність виходу %.3e' % (ours - theirs).abs().max().item())
print('найбільша розбіжність ваг    %.3e' % (weights_ours - weights_theirs).abs().max().item())
assert torch.allclose(ours, theirs, atol=1e-5, rtol=0), 'розрахунок розійшовся!'
print('✅ наша багатоголова увага збігається з nn.MultiheadAttention')

Магії немає: сім рядків арифметики дають те саме, що бібліотечний шар.

## Крок 6 · Доказ, для якого не треба нічого навчати

Тепер найважливіший дослід зошита, і він **не потребує навчання взагалі**.

Твердження: self-attention не бачить порядку слів. Точніше — він
**еквіваріантний щодо перестановки**: якщо переставити слова на вході, вихід не
зміниться, а просто переставиться так само.

Перевірка пряма. Беремо випадкові ваги, подаємо речення, запамʼятовуємо вихід.
Потім переставляємо слова й дивимось: чи дорівнює новий вихід переставленому
старому? Якщо так — шар порядку не помітив.

Причинну маску тут вимикаємо: вона сама по собі залежить від позицій і зіпсувала б
дослід. Ідеться про чистий механізм уваги.

In [ ]:
torch.manual_seed(1)
layer = nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.0)
words = ['не', 'вдалося', 'зберегти', 'файл', 'диск']
ids = torch.tensor([[word_to_id.get(w, UNK) for w in words]])
embed = nn.Embedding(V, D_MODEL)
torch.manual_seed(2)
nn.init.normal_(embed.weight, std=0.5)

permutation = [3, 0, 4, 1, 2]        # нова розсадка слів по місцях

def run(order, add_position):
    vectors = embed(ids[:, order])
    if add_position:
        vectors = vectors + sinusoidal_table(len(order), D_MODEL).unsqueeze(0)
    out, _ = layer(vectors, vectors, vectors, need_weights=False)
    return out

for add_position in (False, True):
    plain = run(list(range(len(words))), add_position)
    mixed = run(permutation, add_position)
    # порівнюємо вихід переставленого речення з переставленим виходом звичайного
    difference = (mixed - plain[:, permutation]).abs().max().item()
    name = 'з позиційним кодуванням' if add_position else 'без позиційного кодування'
    print('%-26s найбільша різниця %.3e' % (name, difference))

plain = run(list(range(len(words))), False)
mixed = run(permutation, False)
assert torch.allclose(mixed, plain[:, permutation], atol=1e-5, rtol=0)
print()
print('✅ без кодування вихід лише переставився — порядку шар не бачить')
print('   з кодуванням різниця на кілька порядків більша: позиція таки дійшла')

Різниця в першому рядку — це шум округлення float32, а не зміна. У другому вона
на кілька порядків більша: позиційне кодування зробило вектори різними ще до входу
в шар.

**Що з цього випливає для мовної моделі.** У неї стоїть причинна маска, тож слово на
позиції `t` бачить лише перші `t` слів. Разом з еквіваріантністю це означає:
без позиційного кодування передбачення залежить від того, **які** слова були раніше,
і зовсім не залежить від їхнього **порядку**. Тобто це мішок слів — той самий, що
в темі про `CountVectorizer`, тільки зроблений мільйоном параметрів.

## Крок 7 · Дві маски, які треба бачити очима

Причинна маска забороняє дивитись праворуч від себе. Маска доповнення забороняє
читати порожні місця, якими вирівняли речення різної довжини в пачці. Обидві
працюють однаково: забороненій клітинці перед softmax додають мінус нескінченність,
і після softmax її вага стає рівно нулем.

Надрукуймо їх, щоб побачити форму.

In [ ]:
steps, real = 6, 4          # шість позицій, з них справжніх чотири
causal = torch.triu(torch.ones(steps, steps, dtype=torch.bool), diagonal=1)
padding = torch.zeros(steps, dtype=torch.bool)
padding[real:] = True

print('заборонено (·) / дозволено (#), рядок = хто дивиться\n')
print('     ' + ' '.join('%2d' % j for j in range(steps)))
for i in range(steps):
    row = []
    for j in range(steps):
        forbidden = bool(causal[i, j]) or bool(padding[j])
        row.append(' ·' if forbidden else ' #')
    print('%3d |' % i, ' '.join(row))

print()
print('без причинної маски рядок 2 бачив би позицію 3 — а там лежить його відповідь')

# і перевіримо, що після softmax заборонене справді дорівнює нулю
scores = torch.randn(steps, steps)
scores = scores.masked_fill(causal | padding[None, :], float('-inf'))
weights = torch.softmax(scores, dim=-1)
print('сума ваг у рядку 3: %.6f' % weights[3].sum().item())
print('вага на забороненій клітинці (3, 4): %.1f' % weights[3, 4].item())

Тепер найважливіша перевірка про маски, і вона **не потребує навчання**.

Твердження: без причинної маски вихід на позиції `t` залежить від входу на позиції
`t + 1`. Якщо це так, то мовна модель бачить власну відповідь.

Перевірка пряма: беремо випадковий шар уваги, подаємо речення, запамʼятовуємо вихід.
Потім міняємо **одне** слово — те, що стоїть праворуч від позиції, за якою стежимо, —
і дивимось, чи змінився вихід на самій позиції. З причинною маскою він змінитись не
може за побудовою; без неї — може.

In [ ]:
torch.manual_seed(3)
probe_layer = nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.0)
sentence = torch.tensor([[EOS, 10, 20, 30, 40, 50]])
watched = 2                      # стежимо за виходом на цій позиції

def output_at(tokens, causal):
    vectors = embed(tokens) + sinusoidal_table(tokens.shape[1], D_MODEL).unsqueeze(0)
    steps = tokens.shape[1]
    mask = torch.triu(torch.ones(steps, steps, dtype=torch.bool), diagonal=1) \
           if causal else None
    out, _ = probe_layer(vectors, vectors, vectors, attn_mask=mask, need_weights=False)
    return out[0, watched]

changed = sentence.clone()
changed[0, watched + 1] = 99      # міняємо СЛОВО ПРАВОРУЧ, тобто відповідь

for causal in (True, False):
    before = output_at(sentence, causal)
    after = output_at(changed, causal)
    name = 'з причинною маскою' if causal else 'без причинної маски'
    print('%-22s зміна виходу на позиції %d: %.3e'
          % (name, watched, (before - after).abs().max().item()))

before = output_at(sentence, True)
after = output_at(changed, True)
assert torch.allclose(before, after, atol=1e-6, rtol=0), 'маска не спрацювала!'
print()
if torch.equal(before, after):
    print('✅ з маскою вихід не змінився ЖОДНИМ бітом — майбутнє недосяжне')
else:
    print('✅ з маскою вихід не змінився в межах точності float32 — майбутнє недосяжне')
print('   без маски змінився: слово праворуч дійшло до виходу, а це і є відповідь')

## Крок 8 · Скільки в пачці порожнечі

Матричні обчислення люблять прямокутники, тому пачку з речень різної довжини
доводиться доповнювати до найдовшого. Скільки при цьому виходить порожнечі —
залежить не від даних, а від того, **як складати пачки**.

Порівняймо два способи: брати речення навмання й брати речення схожої довжини.

In [ ]:
def make_batch(chunk):
    """Пачка з речень різної довжини: короткі доповнюємо нулями до найдовшого."""
    width = max(len(s) for s in chunk) + 1
    x = torch.zeros(len(chunk), width, dtype=torch.long)
    y = torch.zeros(len(chunk), width, dtype=torch.long)
    for row, sentence in enumerate(chunk):
        sequence = [EOS] + sentence + [EOS]
        x[row, :len(sentence) + 1] = torch.tensor(sequence[:-1])
        y[row, :len(sentence) + 1] = torch.tensor(sequence[1:])
    return x, y

def random_groups(data, size, seed):
    order = list(range(len(data)))
    random.Random(seed).shuffle(order)
    return [[data[j] for j in order[i:i + size]] for i in range(0, len(order), size)]

def length_groups(data, size, seed):
    """Пачки з речень схожої довжини: сортуємо за довжиною, самі пачки перемішуємо."""
    order = sorted(range(len(data)), key=lambda j: (len(data[j]), j))
    groups = [[data[j] for j in order[i:i + size]] for i in range(0, len(order), size)]
    if seed is not None:
        random.Random(seed).shuffle(groups)
    return groups

useful = sum(len(s) + 1 for s in fit)
for name, builder in (('навмання', random_groups), ('за довжиною', length_groups)):
    groups = builder(fit, 64, 0)
    cells = sum(len(g) * (max(len(s) for s in g) + 1) for g in groups)
    print('%-12s клітинок %8d, справжніх %8d, порожніх %5.2f %%, роботи ×%.2f'
          % (name, cells, useful, 100 * (1 - useful / cells), cells / useful))

Різниця в кілька разів — і жодна перевірка на неї не поскаржиться, бо обидва
способи правильні. Далі складаємо пачки за довжиною: так навчання швидше, а
порожнечі майже немає.

## Крок 9 · Навчання: функції

Одне навчання — це один прохід по всіх пачках із оптимізатором Adam. Перплексію
рахуємо як експоненту від середньої мінус-логарифмічної ймовірності: вона читається
як «між скількома рівноймовірними словами модель вагається на кожному кроці».

Щоб зошит укладався в кілька хвилин, беремо **8 % навчальної частини**.
Числа від цього виходять гірші, ніж на повному корпусі, — зате однакові умови в усіх
дослідів, а порівнюємо ми саме досліди між собою.

In [ ]:
def perplexity(model, data, causal=True, pad_mask=True, size=128):
    """Експонента середньої мінус-логарифмічної ймовірності на цих даних."""
    model.eval()
    total, counted = 0.0, 0
    with torch.no_grad():
        for group in length_groups(data, size, None):
            x, y = make_batch(group)
            logits = model(x, causal=causal, pad_mask=pad_mask)
            loss = nn.functional.cross_entropy(
                logits.reshape(-1, V), y.reshape(-1),
                ignore_index=PAD, reduction='sum')
            total += loss.item()
            counted += int((y != PAD).sum())
    return math.exp(total / counted)

def train_once(data, seed, lr, nhead=4, use_pos=True, causal=True, batch=64):
    """Одна епоха. Повертає модель і витрачений процесорний час."""
    torch.manual_seed(seed)
    model = TransformerLM(V, nhead=nhead, use_pos=use_pos)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    started = time.process_time()
    for group in length_groups(data, batch, seed):
        x, y = make_batch(group)
        optimizer.zero_grad()
        logits = model(x, causal=causal)
        nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1),
                                    ignore_index=PAD).backward()
        optimizer.step()
    return model, time.process_time() - started

SHARE = 0.08
fit_small = fit[:int(len(fit) * SHARE)]
val_small = val[:2500]           # перевіряємо на частині: зошит має вкластися в час
print('навчаємось на', len(fit_small), 'реченнях із', len(fit))
print('це', sum(len(s) for s in fit_small), 'слововживань')
print('перевіряємо на', len(val_small), 'реченнях із', len(val))

## Крок 10 · Досліди про голови

Тепер головне порівняння зошита: **чотири голови проти однієї**. Параметрів
порівну (ми це надрукували на кроці 4), корпус той самий, зерен **пʼять**.

Чому пʼять, а не три. Зерно міняє початкові ваги й порядок пачок, і розкид від нього
буває більший, ніж здається: на трьох зернах купа виходить помітно вужчою, ніж вона є
насправді, і різниця між моделями через це виглядає переконливішою, ніж вона є.

Швидкість навчання добираємо **на відкладеній вибірці** — окремо для кожного
варіанта, щоб жоден не програв просто через невдалий крок.

In [ ]:
LR_GRID = [0.004, 0.008, 0.016]
holdout_small = holdout[:1200]        # добираємо крок на відкладеній, а не на перевірній
picked = {}
for nhead in (4, 1):
    row = []
    for lr in LR_GRID:
        model, _ = train_once(fit_small, 0, lr, nhead=nhead)
        row.append(perplexity(model, holdout_small))
        del model
        gc.collect()                   # інакше памʼять з-під моделей накопичується
    best = LR_GRID[int(np.argmin(row))]
    picked[nhead] = best
    print('голів %d:' % nhead,
          ' '.join('lr=%-6g %.2f' % (lr, p) for lr, p in zip(LR_GRID, row)))
    print('   дібрано на відкладеній: lr = %g' % best)
print()
print('процесорних секунд від початку зошита: %.0f' % (time.process_time() - NOTEBOOK_START))

In [ ]:
SEEDS = (0, 1, 2, 3, 4)
result = {}
for nhead in (4, 1):
    values, seconds = [], []
    for seed in SEEDS:
        model, spent = train_once(fit_small, seed, picked[nhead], nhead=nhead)
        values.append(perplexity(model, val_small))
        seconds.append(spent)
        if nhead == 4 and seed == 0:
            trained_four_heads = model      # цю модель розберемо на кроках 11-12
        else:
            del model                       # памʼять звільняємо одразу: моделей багато
            gc.collect()
    result[nhead] = values
    print('голів %d, lr=%g: перплексія %.2f (%.2f…%.2f), навчання %.0f с процесорних'
          % (nhead, picked[nhead], float(np.mean(values)),
             min(values), max(values), float(np.mean(seconds))))

low4, high4 = min(result[4]), max(result[4])
low1, high1 = min(result[1]), max(result[1])
overlap = not (high4 < low1 or high1 < low4)
print()
print('різниця середніх: %.2f' % (np.mean(result[1]) - np.mean(result[4])))
print('купи по зернах', 'ПЕРЕТИНАЮТЬСЯ — на цій вибірці різниці НЕ показано' if overlap
      else 'НЕ перетинаються — різниця справжня')
if overlap:
    print()
    print('Це не «голови не потрібні». Це «на', len(fit_small), 'реченнях розкид від')
    print('зерна більший за розрив між моделями». Щоб розрізнити їх, потрібно або')
    print('більше даних, або більше зерен — див. завдання рівня 1.')

## Крок 11 · Чи розходяться голови

Перплексія сказала, що вигідніше. Тепер подивімось, **чому**: чи справді чотири
голови навченої моделі роблять різне.

Мірка проста й наочна: для кожної голови рахуємо, куди йде її вага уваги —
на саму позицію, на сусіда зліва, на два слова назад, далі. Якщо голови роблять те
саме, профілі будуть однакові.

In [ ]:
def attention_of_layer(model, x, layer_index):
    """Ваги уваги кожної голови на вказаному шарі: (пачка, голова, крок, крок)."""
    block = model.encoder.layers[layer_index]
    steps = x.shape[1]
    h = model.embedding(x) * math.sqrt(D_MODEL)
    if model.use_pos:
        h = h + model.pos[:steps].unsqueeze(0)
    future = torch.triu(torch.ones(steps, steps, dtype=torch.bool), diagonal=1)
    padding = (x == PAD)
    with torch.no_grad():
        for earlier in range(layer_index):
            h = model.encoder.layers[earlier](h, src_mask=future,
                                              src_key_padding_mask=padding)
        normalized = block.norm1(h)         # norm_first=True: спершу нормалізація
        _, weights = block.self_attn(normalized, normalized, normalized,
                                     attn_mask=future, key_padding_mask=padding,
                                     need_weights=True, average_attn_weights=False)
    return weights

BUCKETS = ['сама позиція', 'сусід зліва', 'через одне', 'на 3-5 назад', 'далі']

def head_profiles(model, data, n_batches=12):
    heads = model.nhead
    mass = np.zeros((heads, len(BUCKETS)))
    seen = 0
    for group in length_groups(data, 64, 0)[:n_batches]:
        x, _ = make_batch(group)
        weights = attention_of_layer(model, x, 0)
        live = (x != PAD)
        for t in range(1, x.shape[1]):
            rows = live[:, t]
            if rows.sum() == 0:
                continue
            share = weights[rows][:, :, t, :t + 1]        # (речення, голова, ключі)
            offset = torch.arange(t, -1, -1)              # відступ назад від позиції t
            masks = [offset == 0, offset == 1, offset == 2,
                     (offset >= 3) & (offset <= 5), offset >= 6]
            for head in range(heads):
                for bucket, m in enumerate(masks):
                    mass[head, bucket] += share[:, head][:, m].sum().item()
            seen += int(rows.sum())
    return mass / seen

profiles = head_profiles(trained_four_heads, val_small)
print('шар 1, куди йде вага уваги (частки, сума по рядку = 1):\n')
print('        ' + ' '.join('%14s' % b for b in BUCKETS))
for head in range(profiles.shape[0]):
    print('голова %d' % (head + 1),
          ' '.join('%14.4f' % v for v in profiles[head]))
neighbour = profiles[:, 1]
print()
print('вага на сусіда зліва: від %.4f до %.4f, розрив %.4f'
      % (neighbour.min(), neighbour.max(), neighbour.max() - neighbour.min()))

## Крок 12 · Гасимо голову

Профіль показує, що голови різні. Але «різні» ще не означає «потрібні». Прямий
дослід: беремо навчену модель і **вимикаємо** одну голову — обнуляємо той шматок
вихідної матриці, який приймає її внесок. Нічого не донавчаємо, дивимось, наскільки
зросла перплексія.

Якщо голови взаємозамінні, гасіння будь-якої коштуватиме приблизно однаково.

In [ ]:
def knockout(model, data, layer_index):
    """Перплексія моделі з вимкненою кожною головою по черзі."""
    block = model.encoder.layers[layer_index]
    d_head = D_MODEL // model.nhead
    values = []
    for head in range(model.nhead):
        columns = slice(head * d_head, (head + 1) * d_head)
        weight = block.self_attn.out_proj.weight
        saved = weight[:, columns].clone()
        with torch.no_grad():
            weight[:, columns] = 0
        values.append(perplexity(model, data))
        with torch.no_grad():
            weight[:, columns] = saved       # обовʼязково повертаємо назад
    return values

base = perplexity(trained_four_heads, val_small)
damage = knockout(trained_four_heads, val_small, 0)
print('ціла модель: %.2f\n' % base)
for head, value in enumerate(damage):
    print('без голови %d: %8.2f   (+%.2f)' % (head + 1, value, value - base))
print()
print('найдорожча голова коштує +%.2f, найдешевша +%.2f, розрив %.2f'
      % (max(damage) - base, min(damage) - base, max(damage) - min(damage)))

## Крок 13 · Що буде без причинної маски

Останній дослід — і найпоказовіший. Прибираємо причинну маску, більше не міняємо
нічого. Тепер позиція `t` бачить позицію `t+1`, а там лежить слово, яке вона мусить
угадати.

Перплексію теж рахуємо без маски: інакше ми міряли б не ту модель, яку навчили.

Наперед скажу, чого чекати. Число впаде — але не обовʼязково до одиниці. Щоб
скористатися витоком, моделі мало **бачити** сусіда праворуч: їй ще треба
**навчитись** туди дивитись, а з синусоїдним кодуванням «одне слово праворуч» — це
поворот, який матриці запиту й ключа мусять вивести самі. За один прохід по маленькому
корпусу вони встигають це лише частково. Витік від цього не перестає бути витоком:
задача зіпсована в момент, коли ми дали моделі побачити відповідь.

In [ ]:
leaky = []
for seed in (0, 1, 2):
    model, _ = train_once(fit_small, seed, picked[4], nhead=4, causal=False)
    leaky.append(perplexity(model, val_small, causal=False))
    del model
    gc.collect()
print('без причинної маски: %8.2f  (%.2f…%.2f) на трьох зернах'
      % (float(np.mean(leaky)), min(leaky), max(leaky)))
print('з причинною маскою:  %8.2f  (%.2f…%.2f) на пʼятьох зернах'
      % (float(np.mean(result[4])), low4, high4))
print()
print('без маски вийшло у %.2f раза «краще» — але це не якість моделі.'
      % (np.mean(result[4]) / np.mean(leaky)))
print('Позиція t бачить позицію t+1, а там лежить слово, яке вона мусить угадати.')
print('Задача перетворилась на копіювання, тож це число з чесним не порівнюване.')
print('Що довше вчити таку модель, то ближче її перплексія до одиниці —')
print('до стану «жодних сумнівів», бо відповідь просто списують.')

## Що ми з цього дізнались

Три заміри, три відповіді.

1. **Голови.** Чотири голови проти однієї при однаковій кількості параметрів.
   Дивись не на різницю середніх, а на те, чи перетинаються купи по зернах: різниця,
   менша за розкид, різницею не є. На восьми відсотках корпусу купи, найімовірніше,
   перетнулись — і це чесна відповідь «не показано», а не «різниці немає». Профілі
   голів при цьому вийшли різні, і гасіння різних голів коштує по-різному: тобто
   голови таки роблять не одне й те саме, просто на маленькій вибірці це не
   встигає перетворитись на виграш у перплексії.
2. **Порядок слів.** Доведено без навчання: перестановка слів переставляє вихід, а
   не міняє його. Це властивість механізму, а не результат прогону.
3. **Причинна маска.** Без неї перплексія обвалюється — і це не перемога, а
   зіпсована постановка задачі: модель читає відповідь із власного входу.
   Підозріло добре число перевіряй першим.

І окремо — про час. Ми весь час міряли **процесорний** час, а не стінний: на
завантаженій машині стінний годинник показує втричі більше й нічого не означає.

In [ ]:
spent = time.process_time() - NOTEBOOK_START
print('усього процесорного часу: %.0f с (%.1f хв)' % (spent, spent / 60))
print('навчань у зошиті: %d' % (len(LR_GRID) * 2 + len(SEEDS) * 2 + 3))

## Завдання

### 🟢 Рівень 1 — База

Постав `SHARE = 0.3` замість 0.08 і повтори порівняння «4 голови проти 1»
(зерен лишай пʼять). Зошит рахуватиме довше — близько чверті години. Питання одне:
**чи розійшлися купи**, коли даних стало вчетверо більше?

**Зроблено, якщо:** названо обидві купи до й після збільшення вибірки, сказано, чи
вони перетинаються в кожному випадку, і словами пояснено, що саме змінилось — самі
моделі чи наша здатність їх розрізнити.

### 🟡 Рівень 2 — Плюс

Замінь синусоїдне позиційне кодування на **навчувану таблицю**
(`nn.Embedding(max_len, D_MODEL)`, додається так само). Заміряй перплексію на
пʼятьох зернах і порівняй із синусоїдним.

**Зроблено, якщо:** названо обидві купи по зернах і сказано, чи вони перетинаються.
Додатково поясни, що станеться з кожним із двох варіантів, якщо на вхід подати
речення довше за `max_len`.

### 🔴 Рівень 3 — Виклик

Перевір, чи розходяться голови **другого** шару так само, як першого: побудуй для
нього профілі й гасіння. Потім зроби те, чого зошит не робив, — заміряй
**попарне розходження Єнсена — Шеннона** між розподілами уваги голів і порівняй
його з тим самим числом на **ненавченій** моделі.

**Зроблено, якщо:** названо середнє розходження на навченій і ненавченій моделі й
зроблено висновок, чи розходження здобуте навчанням, чи воно просто лишилось від
випадкового старту.

## Підказки

- Купа по зернах — це просто мінімум і максимум усіх зерен. Якщо відрізки двох
  моделей накладаються хоч частково, різниці немає.
- Гасячи голову, не забудь **повернути** ваги назад, інакше наступний замір
  міряє вже покалічену модель.
- Навчувану таблицю позицій треба додавати **до** входу першого шару, а не в
  кожному шарі: у класичному трансформері позиція подається рівно один раз.